In [ ]:
from models import DeepUNet
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [2]:
in_dir = "../res_q/"
out_dir = "../res_q/"
channels = 12
alli_yn = 0

In [3]:
print("Loading losses...")
losses = np.load(in_dir + "losses.npy")
best_epoch = np.argmin(losses)

Loading losses...


In [ ]:
# Model
device = torch.device("cuda")

# Set up model
model = DeepUNet(in_channels=channels, out_channels=1)
model = model.to(device)
model = nn.DataParallel(model)

# model.load_state_dict(torch.load(in_dir+"epoch_{}".format(np.argmin(losses)), map_location=torch.device('cuda'))["model_state_dict"])
model.load_state_dict(
    torch.load(in_dir + "epoch", map_location=torch.device("cuda"))["model_state_dict"]
)
model.eval()

In [5]:
# high:1672-1678;967-1021
# low:1813-1824;1959-1970
target = np.load("../data/test/mbmp/967.npy")
context = np.load("../data/test/s2/967.npy")

In [6]:
s = 64
mid_loc_x = target.shape[0] // 2
mid_loc_y = target.shape[1] // 2

target = target[mid_loc_x - s : mid_loc_x + s, mid_loc_y - s : mid_loc_y + s]
context = context[mid_loc_x - s : mid_loc_x + s, mid_loc_y - s : mid_loc_y + s, :]
target = 0.2989 * target[:, :, 0] + 0.5870 * target[:, :, 1] + 0.1140 * target[:, :, 2]

In [7]:
out = model(
    torch.from_numpy(context).float().to(device).permute(2, 0, 1).unsqueeze(0) / 255
)

In [8]:
out.shape

torch.Size([1, 128, 128, 1])

In [ ]:
out_n = out.detach().cpu()[0, :, :, 0]
plt.imshow(out_n)
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(target / 255)
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(out_n - target / 255)
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(context[:, :, :3])
plt.show()